# Agentic RAG on the Week 6 Corpus

## Overview
Plain RAG is a fixed pipeline: **retrieve once, then generate**. It fails quietly when the first retrieval is poor (the model answers from irrelevant text), when a question needs several lookups, or when the answer is simply not in the knowledge base (the model should say so).

**Agentic RAG** hands the control flow to the agent. Retrieval becomes a *tool*, and the agent decides whether to search, what to search for, whether the results are good enough, whether to rewrite the query and try again, and when to stop and say "I don't know".

This notebook compares three designs on the **same corpus and the same questions you used in week 6**:

| Design | Control flow | Who decides |
|:--|:--|:--|
| **Naive RAG** | retrieve, generate | Fixed code |
| **Corrective RAG (CRAG)** | retrieve, **grade** the passages, rewrite the query and retry if they are poor, generate | Fixed code with a feedback loop |
| **Agentic RAG** | a Strands agent with a `search_kb` tool | The model |

### Why this corpus
In week 6 you built a RAG pipeline on **HotpotQA**, a question-answering dataset made of Wikipedia paragraphs. Using the same data here has three benefits:

1. **The questions are multi-hop.** To answer "What university did the last Detroit Pistons player to wear the number retired for 'The Worm' attend?", you first have to learn who "The Worm" is and which number was retired, then who wore it last, then where that player studied. One retrieval with the whole question rarely finds all of it. This is exactly where an agent that can search, read, and search again should help.
2. **We know the right passages.** Every question comes with its two *gold* passages, so besides "is the answer right?" we can ask "did the system ever retrieve both pieces of evidence?".
3. **The numbers are comparable.** The corpus, the embeddings, the 15 evaluation questions and the scoring are the ones from week 6, and at the end we add this notebook's results to the same ledger.

```text
Naive RAG      question ---> retrieve (top 3) ---> generate

CRAG           question ---> retrieve (top 5) ---> grade ---> any relevant? --yes--> generate
                                  ^                              | no
                                  +------- rewrite the query <---+   (at most 2 retries)

Agentic RAG    question ---> AGENT <---> search_kb(query)   (as many searches as it decides, up to a budget)
                               |
                               +---> short answer, or "not available"
```

The notebook uses one agent with one retrieval tool (`search_kb`) over the Qdrant collection from week 6. Strands features covered: `@tool` and a trace of every tool call.

## Setup and prerequisites

You need four things running or available:

* **The chat model** named in `config.yaml` under `rag.model` (`ollama pull gemma4:12b-mlx`). An agent that must call a tool several times needs a capable model. A small one often skips the second search or gives up too early, so this notebook uses a larger model than the other week 7 notebooks.
* **The Qdrant server** from week 6 (`docker compose up -d` in `code/week6`, or `docker start qdrant`).
* **The `hotpotqa_passages` collection**, which `6.5-Storage-Vector-Databases.ipynb` creates. It holds 997 Wikipedia paragraphs.
* **The week 6 packages** (`pip install -r requirements.txt` here also installs `sentence-transformers`, `langchain-huggingface`, `langchain-qdrant` and `datasets`).

Queries are embedded with the **same model as week 6** (`sentence-transformers/all-mpnet-base-v2`). This matters: vectors from different embedding models live in different spaces, so searching the week 6 collection with a different model would return nonsense.

Settings are in `config.yaml` under `rag:` (the model, `top_k`, the number of questions, and where the week 6 folder is). Everything else about the corpus comes from `code/week6/config.yaml`.

In [ ]:
#!pip install -r requirements.txt

In [ ]:
import json, logging, re, sys, time, warnings
from pathlib import Path

import pandas as pd
import yaml
from strands import Agent, tool
from strands.models.ollama import OllamaModel

warnings.filterwarnings("ignore")
logging.getLogger("strands").setLevel(logging.WARNING)

cfg = yaml.safe_load(open("config.yaml"))
rag = cfg["rag"]

def make_model(**overrides):
    """One place to build the chat model: reproducible (temperature 0), fast (no hidden "thinking" tokens),
    and bounded: answers are a few words, so replies are capped at 300 tokens and a call that hangs fails after 90 s."""
    params = dict(
        host=cfg["ollama"]["url"],
        model_id=rag["model"],
        temperature=0,
        keep_alive="1h",                       # keep the model loaded between the many calls below
        additional_args={"think": False},
        max_tokens=300,
        ollama_client_args={"timeout": 90},
        options={"num_ctx": cfg["strands"]["num_ctx"]},
    )
    params.update(overrides)
    return OllamaModel(**params)

llm = make_model()
print("Chat model:", llm.config["model_id"], "@", cfg["ollama"]["url"])

## The corpus: HotpotQA from week 6

In `6.5` you sampled **100 questions** from the HotpotQA validation set. Each question comes with 10 paragraphs: **2 gold paragraphs** that contain the evidence, and **8 distractors** that look related but do not help. Pooled and de-duplicated, that is **997 paragraphs**, now stored in Qdrant. All the questions we use are *bridge* questions: the answer needs a fact from one paragraph and a second fact from another.

We reuse the helper module `rag_common.py` from week 6. It gives us the configuration, the question list, the shared evaluation sample and the scoring functions, so that nothing here is re-implemented differently.

The question list and the paragraph list are cached by `6.5` in `/tmp`, and `/tmp` is cleared when your computer restarts. The next cell reads the cache and, if it is gone, rebuilds it with exactly the same recipe and random seed as `6.5` (the dataset is downloaded once and then kept in your Hugging Face cache). The Qdrant collection itself is not touched.

In [ ]:
sys.path.insert(0, str((Path.cwd() / rag["week6_dir"]).resolve()))
import rag_common as rc

cfg6 = rc.load_config()                      # code/week6/config.yaml: Qdrant URL, embedding model, HotpotQA settings
rc.check_services(cfg6, llm=False)           # prints whether Qdrant is reachable

def rebuild_hotpotqa_caches(cfg6):
    """Recreate what 6.5 caches: the same sample (same split and seed) of HotpotQA questions and paragraphs."""
    import random
    from datasets import load_dataset
    h = cfg6["hotpotqa"]
    ds = load_dataset("hotpot_qa", "distractor", split=h.get("split", "validation"))
    random.seed(h.get("random_seed", 42))
    sample = [ds[i] for i in random.sample(range(len(ds)), k=min(h.get("sample_size", 100), len(ds)))]
    qa_pairs = [{"question": ex["question"], "answer": ex["answer"], "type": ex["type"], "level": ex["level"],
                 "supporting_titles": list(dict.fromkeys(ex["supporting_facts"]["title"]))} for ex in sample]
    seen, corpus = set(), []
    for ex in sample:
        gold = set(ex["supporting_facts"]["title"])
        for title, sentences in zip(ex["context"]["title"], ex["context"]["sentences"]):
            text = " ".join(sentences).strip()
            if title not in seen and text:
                seen.add(title)
                corpus.append({"id": f"{title}#0", "title": title, "text": text, "is_supporting": title in gold})
    return qa_pairs, corpus

try:
    qa_pairs, passages = rc.load_hotpotqa(cfg6)
    print("Loaded the week 6 caches.")
except FileNotFoundError:
    print("Caches not found (/tmp was cleared?). Rebuilding them from the dataset ...")
    qa_pairs, passages = rebuild_hotpotqa_caches(cfg6)

client, embeddings, store = rc.connect_qdrant(cfg6)     # same collection and same embedding model as week 6
print(f"{len(qa_pairs)} questions | {len(passages)} paragraphs | "
      f"{client.count(cfg6['hotpotqa']['collection']).count} vectors in Qdrant")

## The test questions

We use two groups of questions.

**Answerable (10).** `rc.eval_set` returns the fixed sample of 15 that every week 6 notebook is scored on. We take the first 10 so the run stays short (set `rag.n_questions: 15` for the full sample). Each has a short reference answer and two gold paragraphs.

**Unanswerable (3).** A good system must also know when the knowledge base cannot help. These three ask about companies that do not exist, so the model cannot know the answer from memory and the corpus cannot contain it. The correct behaviour is to say the information is not available.

In [ ]:
N = rag["n_questions"]
QUESTIONS = [
    {"question": e["question"], "answer": e["answer"], "type": e["type"], "gold": e["supporting_titles"]}
    for e in rc.eval_set(qa_pairs, n=15)[:N]      # the first N of the shared week 6 sample of 15
]
UNANSWERABLE = [
    {"question": q, "answer": None, "type": "unanswerable", "gold": [], "abstain": True}
    for q in [
        "In what year was the Brightwave Appliances company founded?",
        "Who is the chief executive of Norvik Machines?",
        "How many people work at the Lumora Robotics factory in Gdansk?",
    ]
]
ALL_QUESTIONS = QUESTIONS + UNANSWERABLE

pd.DataFrame([
    {"#": i, "type": q["type"], "question": q["question"][:95], "reference": q["answer"] or "(abstain)",
     "gold paragraphs": " + ".join(q["gold"])}
    for i, q in enumerate(ALL_QUESTIONS, 1)
]).set_index("#")

## The retrieval function

Every design uses this one function: embed the query, ask Qdrant for the nearest paragraphs. Each hit carries its **title** (the Wikipedia page it came from) and a **score** (cosine similarity, higher is closer). The `n` field numbers the hits `1..k` so that the grader and the generator can refer to them.

Try it on the first question. The check mark shows which hits are gold paragraphs. Notice that the question never names the player, so a single retrieval has to find the evidence by meaning alone.

In [ ]:
TOP_K = rag["top_k"]

def retrieve(query: str, k: int = TOP_K) -> list[dict]:
    hits = store.similarity_search_with_score(query, k=k)
    return [{"n": i, "title": d.metadata["title"], "text": d.page_content, "score": round(float(s), 3)}
            for i, (d, s) in enumerate(hits, 1)]

demo = QUESTIONS[0]
print("Q:", demo["question"], "\nGold:", demo["gold"], "\n")
for h in retrieve(demo["question"]):
    print(("gold  " if h["title"] in demo["gold"] else "      "), h["n"], h["score"], h["title"])

## Design 1: Naive RAG

Retrieve the top 3 paragraphs once, put them in a prompt, generate. The prompt tells the model to answer **only** from the context, to reply with just the answer (HotpotQA answers are a few words), and to say so if the answer is missing.

This is the baseline from week 6. Keep an eye on what the pipeline cannot do: if the retrieved paragraphs are the wrong ones, the generator still has to answer from them, and nothing in the code reacts.

In [ ]:
NOT_AVAILABLE = "The information is not available in the knowledge base."

ANSWER_PROMPT = """Answer the question using ONLY the context. Reply with only the answer itself: a few words, no explanation.
If the context does not contain the answer, reply exactly: """ + NOT_AVAILABLE + """

Context:
{context}

Question: {question}"""

def generate(question, passages):
    context = "\n\n".join(f"[{p['n']}] {p['title']}: {p['text']}" for p in passages) or "(no context)"
    agent = Agent(model=llm, callback_handler=None)          # a fresh agent per call: no memory of earlier questions
    return str(agent(ANSWER_PROMPT.format(context=context, question=question))).strip()

def naive_rag(question):
    passages = retrieve(question)
    return generate(question, passages), {"searches": 1, "seen": passages}

# A question where the first retrieval is easy to get wrong. The reference answer is shown for comparison.
item = QUESTIONS[1]
answer, info = naive_rag(item["question"])
print("Q:", item["question"])
print("retrieved:", [p["title"] for p in info["seen"]], "| gold:", item["gold"])
print("A:", answer, "| reference:", item["answer"])

## Design 2: Corrective RAG (CRAG)

CRAG adds a feedback loop around retrieval, from the paper *Corrective Retrieval Augmented Generation*. After retrieving 5 paragraphs:

1. A **grader** judges which paragraphs are relevant to the question.
2. If none is relevant, **rewrite** the query into a retrieval-friendly form and retrieve again (at most 2 retries).
3. **Generate** from the relevant paragraphs only. If nothing relevant was ever found, abstain.

CRAG retrieves 5 instead of 3 on purpose: it can afford to over-fetch because the grader throws away the noise.

The grader answers with just the numbers of the relevant paragraphs (or the word `none`), and the code parses them. Strands also offers *structured output*, where the model must return a typed object, which is neater when it works. We deliberately do not use it here: with local models it can retry for a very long time when the right answer is an empty list, which is exactly the case CRAG has to handle. A grader built on simple text that you parse yourself is slower to write but cannot hang.

In [ ]:
def grade(question, passages) -> list[dict]:
    """Ask the model which paragraphs help. It replies with numbers, or "none"; we parse the reply."""
    listing = "\n\n".join(f"[{p['n']}] {p['title']}: {p['text']}" for p in passages)
    prompt = (f"Question: {question}\n\nParagraphs:\n{listing}\n\n"
              "Which paragraphs contain information needed to answer the question? "
              "Reply with ONLY their numbers separated by commas, or the single word none.")
    reply = str(Agent(model=llm, callback_handler=None)(prompt)).strip()
    if re.match(r"\W*none", reply, re.I):
        return []
    keep = {int(x) for x in re.findall(r"\d+", reply)}
    return [p for p in passages if p["n"] in keep]

def rewrite(question) -> str:
    prompt = ("Rewrite this question as a short search query with the specific names a Wikipedia article would contain. "
              "Output only the query.\n\nQuestion: " + question)
    return str(Agent(model=llm, callback_handler=None)(prompt)).strip().strip('"')

def corrective_rag(question, max_retries=2, k=5):
    query, trace, seen = question, [], []
    for attempt in range(max_retries + 1):
        passages = retrieve(query, k)
        seen += passages
        good = grade(question, passages)
        trace.append({"query": query, "retrieved": [p["title"] for p in passages], "kept": [p["title"] for p in good]})
        if good:
            return generate(question, good), {"searches": attempt + 1, "seen": seen, "trace": trace}
        query = rewrite(question)
    return NOT_AVAILABLE, {"searches": max_retries + 1, "seen": seen, "trace": trace}

answer, info = corrective_rag(item["question"])
for step in info["trace"]:
    print(step)
print("A:", answer, "| reference:", item["answer"])

## Design 3: Agentic RAG

Now remove the fixed loop. Retrieval is a **tool**, and the model decides what to do. The system prompt is the policy. It tells the agent that questions usually need two facts, that it should search for the first, read the result, and then search again *using the name it just learned*, and that it may only abstain after trying different queries.

Two details make this safe and measurable:

* **A search budget.** The tool refuses after 4 searches, so a confused model cannot loop forever. Together with the token cap and the timeout in `make_model`, nothing in this notebook can run away.
* **A trace.** Every query and its results are recorded in `SEARCHES`. After a run we can read exactly what the agent did, and we use it to check whether it ever retrieved both gold paragraphs.

The tool returns each paragraph's similarity score as well, which gives the model a hint about how good a hit is.

In [ ]:
SEARCHES = []        # trace of every query the agent issues
MAX_SEARCHES = 4

@tool
def search_kb(query: str) -> str:
    """Search the knowledge base of Wikipedia paragraphs and return the most relevant ones.

    Args:
        query: A short search query with specific names. Not a full-sentence question.
    """
    if len(SEARCHES) >= MAX_SEARCHES:
        return "Search budget used up. Answer now from what you have, or say the information is not available."
    hits = retrieve(query)
    SEARCHES.append({"query": query, "titles": [h["title"] for h in hits], "passages": hits})
    return "\n\n".join(f"[{h['n']}] {h['title']} (score {h['score']}): {h['text']}" for h in hits)

AGENT_PROMPT = """You answer questions using ONLY the search_kb tool. Never use outside knowledge.

How to work:
1. Questions often need two facts: one that leads to something named, and a second about that named thing. Search for the first fact, read the paragraphs, then search again using the name you found.
2. Use short queries with specific names, not whole questions. If the results are poor, rewrite the query with different words. Make at most 4 searches.
3. When you have the answer, reply with ONLY the answer: a few words, no explanation, no sources.
4. Only if several different searches still do not contain the answer, reply exactly: """ + NOT_AVAILABLE

def agentic_rag(question):
    SEARCHES.clear()
    agent = Agent(model=llm, system_prompt=AGENT_PROMPT, tools=[search_kb], callback_handler=None)
    answer = str(agent(question)).strip()
    seen = [p for s in SEARCHES for p in s["passages"]]
    return answer, {"searches": len(SEARCHES), "seen": seen, "trace": [{"query": s["query"], "retrieved": s["titles"]} for s in SEARCHES]}

answer, info = agentic_rag(item["question"])
for step in info["trace"]:
    print("search:", step["query"], "->", step["retrieved"])
print("A:", answer, "| reference:", item["answer"])

Read the trace. On a multi-hop question a good run searches for the first fact, reads the result, and then searches again for the name it found. A fixed pipeline retrieves once with the whole question and has no second step.

## Compare the three designs

Now we run all 13 questions through all three designs. This takes about three to five minutes, because the agentic design makes several model calls per question. The progress line shows where it is. For each answer we record:

| Metric | Meaning |
|:--|:--|
| **correct** | The answer matches the reference (token F1 of at least 0.5, or the reference appears in the answer). For an unanswerable question: the system abstained. A made-up answer to an unanswerable question counts as wrong. |
| **both gold found** | Both gold paragraphs were retrieved at some point, even if the final answer was wrong. This separates *retrieval* failures from *generation* failures. |
| **hit** and **F1** | The same two numbers the week 6 ledger uses: the reference answer appears in the retrieved text, and the token-level F1 of the answer. |
| **searches** | Number of retrieval calls: a proxy for cost. |
| **seconds** | Wall-clock time per question. |

In [ ]:
ABSTAIN = re.compile(r"(not (available|mentioned|provided|found|stated|specified|included)|no information|"
                     r"don't know|do not know|cannot (find|answer)|unable to)", re.I)

def is_correct(item, answer):
    if item.get("abstain"):
        return bool(ABSTAIN.search(answer))
    if ABSTAIN.search(answer):                     # it gave up although the answer exists
        return False
    ref = item["answer"]
    return rc.token_f1(answer, ref) >= 0.5 or rc.normalize_text(ref) in rc.normalize_text(answer)

DESIGNS = {"naive": naive_rag, "crag": corrective_rag, "agentic": agentic_rag}

rows, answers = [], {}
for i, item in enumerate(ALL_QUESTIONS, 1):
    for name, run in DESIGNS.items():
        t0 = time.time()
        try:
            answer, info = run(item["question"])
        except Exception as exc:                  # a timed-out or failed call counts as a failure, not a crash
            answer, info = f"(error: {type(exc).__name__})", {"searches": 0, "seen": []}
        seen_titles = {p["title"] for p in info["seen"]}
        seen_text = rc.normalize_text(" ".join(p["text"] for p in info["seen"]))
        answerable = not item.get("abstain")
        rows.append({
            "q": i, "type": item["type"], "design": name,
            "correct": is_correct(item, answer),
            "both_gold": set(item["gold"]) <= seen_titles if answerable else None,
            "hit": float(rc.normalize_text(item["answer"]) in seen_text) if answerable else None,
            "f1": rc.token_f1(answer, item["answer"]) if answerable else None,
            "searches": info["searches"], "seconds": round(time.time() - t0, 1),
        })
        answers[(i, name)] = answer
    print(f"question {i}/{len(ALL_QUESTIONS)} done", end="\r")
results = pd.DataFrame(rows)
print("done" + " " * 20)

In [ ]:
answerable = results[results["type"] != "unanswerable"]
unanswerable = results[results["type"] == "unanswerable"]

summary = pd.DataFrame({
    f"Correct (of {len(QUESTIONS)})": answerable.groupby("design")["correct"].sum(),
    f"Both gold found (of {len(QUESTIONS)})": answerable.groupby("design")["both_gold"].sum(),
    f"Abstained correctly (of {len(UNANSWERABLE)})": unanswerable.groupby("design")["correct"].sum(),
    "Avg searches": results.groupby("design")["searches"].mean().round(1),
    "Avg seconds": results.groupby("design")["seconds"].mean().round(1),
}).loc[["naive", "crag", "agentic"]]
summary

Local models are not fully deterministic, so your table can differ by a question or two. Read the actual answers, not only the scores: a keyword check can miss a correct paraphrase and can accept a lucky guess.

The next cell lists every question with the three answers side by side. Compare *both gold found* with *correct*: if the gold paragraphs were retrieved but the answer is wrong, the generator is to blame. If they were not retrieved, retrieval is.

In [ ]:
for i, item in enumerate(ALL_QUESTIONS, 1):
    print(f"Q{i} [{item['type']}]: {item['question']}")
    print(f"     reference: {item['answer'] or '(should abstain)'}")
    for name in DESIGNS:
        r = results[(results.q == i) & (results.design == name)].iloc[0]
        gold = "" if r.both_gold is None else f" gold={'yes' if r.both_gold else 'no '}"
        print(f"   {'OK  ' if r.correct else 'FAIL'} {name:8s} searches={r.searches}{gold} | {answers[(i, name)][:110]!r}")
    print()

## Add the results to the week 6 ledger

Week 6 collected one row per technique into a shared table (`rag_common.record_result`). We add the three designs here, using exactly the same two numbers: **Hit %** (the reference answer appears in the retrieved text) and **F1 %** (token overlap between the answer and the reference), measured on the same questions as week 6 (the first `n_questions` of the shared sample). Only answerable questions are included, because those two numbers need a reference answer.

If you ran the week 6 notebooks in this session, their rows appear in the same table, so you can set the agent against the best fixed pipeline from `6.13`.

In [ ]:
for name, label in [("naive", "7.6 Naive RAG (top 3)"), ("crag", "7.6 Corrective RAG"), ("agentic", "7.6 Agentic RAG")]:
    d = answerable[answerable.design == name]
    rc.record_result(label, "agentic-rag", rows=[{"hit": h, "f1": f} for h, f in zip(d["hit"], d["f1"])])
rc.show_ablation()

## When to use which

| | Naive RAG | CRAG | Agentic RAG |
|:--|:--|:--|:--|
| Latency and cost | Lowest: 1 retrieval, 1 generation | Medium: grading adds model calls | Highest and variable: several model turns |
| Handles a bad first retrieval | No | **Yes**, by rewriting | Yes |
| Multi-hop questions | No | Weak: one query for the whole question | **Yes**: searches again with what it learned |
| Abstains when the answer is missing | Only if the prompt works | **Yes**, by design | Yes, by prompt |
| Predictability | **High** | High | Lower: the model chooses the path |
| Needs a capable model | No | No | **Yes**: a small model skips searches or gives up early |

Rules of thumb:

* Start with naive RAG and **measure it**. Often it is good enough.
* Add CRAG when retrieval quality is the problem and questions are single-step.
* Move to agentic RAG when questions need several lookups, several sources, or several tools, and when you can afford the extra seconds and a larger model.
* Evaluate every change with a fixed question set, as week 6 taught. The table above is that habit in miniature.

## Summary

1. RAG failures are mostly **retrieval** failures that the generator hides. Comparing *both gold found* with *correct* tells you which part failed.
2. CRAG puts a **grade and rewrite** feedback loop around retrieval, in fixed code.
3. Agentic RAG makes retrieval a **tool** and lets the model plan: search, read, search again with a new name, rewrite, or abstain.
4. More autonomy costs latency and predictability, and it needs a more capable model. Choose it when the question mix needs it, and always measure.

## Try it yourself

* Set `rag.n_questions` to 15 in `config.yaml` for the full week 6 sample, or change `rag.top_k` to 5, and rerun. Does naive RAG close the gap with the agent?
* Lower `MAX_SEARCHES` to 2. Which questions fail first?
* Swap `rag.model` for a smaller model (for example `gemma4:e4b-mlx`). Does the agent still beat naive RAG? Read its traces to see why.
* Add a second tool, `get_passage(title)`, that returns a whole paragraph by its title. Does the agent use it?